## Prepare a jsonl dataset file

In [ ]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import mlcroissant as mlc

hint_path = Path("data/nanogpt_speedrun_knowledge_in_levels")
workspace_path = Path("workspace_templates/nanogpt_speedrun")

target_hint_path = Path("hints")
target_workspace_path = Path("workspaces")

records = []
for level in ["0", "1", "2", "3", "1+2", "1+2+3"]:
    for i in range(1, 21):
        record_info = {
            "id": i,
            "current_results": (workspace_path / f"record_{i}" / "results.json").read_text(),
            "current_train_script": (workspace_path / f"record_{i}" / "train_gpt2.py").read_text(),
            "current_results_path": str(target_workspace_path / f"record_{i}" / "results.json"),
            "current_train_script_path": str(target_workspace_path / f"record_{i}" / "train_gpt2.py"),
            "target_time": json.loads(
                (workspace_path / f"record_{i+1}" / "results.json").read_text()
            )["metrics"]["train_time"],
        }
        hints = []
        hint_paths = []
        if "1" in level:
            hints.append(
                (hint_path / f"record_{i}" / "level_1_pseudo.txt").read_text()
            )
            hint_paths.append(
                str(target_hint_path / f"record_{i}" / "level_1_pseudo.txt")
            )
        if "2" in level:
            hints.append(
                (hint_path / f"record_{i}" / "level_2_description.txt").read_text()
            )
            hint_paths.append(
                str(target_hint_path / f"record_{i}" / "level_2_description.txt")
            )
        if "3" in level:
            hints.append(
                (hint_path / f"record_{i}" / "level_5_paper.txt").read_text()
            )
            hint_paths.append(
                str(target_hint_path / f"record_{i}" / "level_5_paper.txt")
            )
        record_info["hints"] = hints
        record_info["hint_paths"] = hint_paths
        records.append(record_info)

records_df = pd.DataFrame(records)
records_df.to_json("data.jsonl", orient="records", lines=True)

## Generate a Croissant metadata file

In [3]:
# FileObjects and FileSets define the resources of the dataset.
distribution = [
    mlc.FileObject(
        id="llm_speedrun_records.zip",
        name="llm_speedrun_records.zip",
        description="Zip file containing the LLM Speedrunning Benchmark dataset",
         # TODO: change hosted zip location
        content_url=(
            "https://drive.usercontent.google.com/download?id=1bBgbb8mt5liZmAuebYyznN1rUcG4ZRe9&confirm=xxx"
        ),
        encoding_formats=["application/zip"],
        sha256="ba45434ce5c947a2086b717351f18e21d01f7d9239f3ce549dbe4db3e2e0551d",
    ),
    mlc.FileObject(
        id="data",
        name="data",
        description="",
        contained_in=["llm_speedrun_records.zip"],
        content_url="scientist/data.jsonl",  # TODO: change path
        encoding_formats=["application/jsonlines"],
    ),
]
record_sets = [
    # RecordSets contains records in the dataset.
    mlc.RecordSet(
        id="llm_speedrun_records",
        name="llm_speedrun_records",
        # Each record has one or many fields...
        fields=[
            # Fields can be extracted from the FileObjects/FileSets.
            mlc.Field(
                id="llm_speedrun_records/id",
                name="id",
                description="The nanoGPT speedrun record ID",
                data_types=mlc.DataType.INTEGER,
                source=mlc.Source(
                    file_object="data",
                    extract=mlc.Extract(column="id"),
                ),
            ),
            *[
                mlc.Field(
                    id=f"llm_speedrun_records/{field_name}",
                    name=field_name,
                    description=description,
                    data_types=mlc.DataType.TEXT,
                    source=mlc.Source(
                        file_object="data",
                        # Extract the field from the column of a FileObject/FileSet:
                        extract=mlc.Extract(column=field_name),
                    ),
                ) for field_name, description in {
                    "current_results": "JSON output for the current record",
                    "current_train_script": "Python script used to train the model for the current record",
                    "current_results_path": "The local path to the JSON output for the current record",
                    "current_train_script_path": "The local path to the Python script used to train the model for the current record",
                }.items()
            ],
            *[
                mlc.Field(
                    id=f"llm_speedrun_records/{field_name}",
                    name=field_name,
                    description=description,
                    data_types=mlc.DataType.TEXT,
                    is_array=True,
                    source=mlc.Source(
                        file_object="data",
                        # Extract the field from the column of a FileObject/FileSet:
                        extract=mlc.Extract(column=field_name),
                    ),
                ) for field_name, description in {
                    "hints": "List of hints for the next record", 
                    "hint_paths": "List of local paths to the hint files for the next record",
                }.items()
            ],
            mlc.Field(
                id="llm_speedrun_records/target_time",
                name="target_time",
                description="The target runtime that the next record achieved",
                data_types=mlc.DataType.INTEGER,
                source=mlc.Source(
                    file_object="data",
                    extract=mlc.Extract(column="target_time"),
                ),
            ),
        ],
    )
]

# Metadata contains information about the dataset.
metadata = mlc.Metadata(
    name="llm_speedrun",
    # Descriptions can contain plain text or markdown.
    description=(
        "This dataset contains the LLM Speedrunning Benchmark, a series of records "
        "from the nanoGPT speedrun accompanied by hints describing how to reach the next record. "
        "The goal of this benchmark is to evaluate the ability of AI agents to ingest knowledge and "
        "implement improvements to LLM training algorithms."
    ),
    cite_as=(
        "@article{<TO_FILL_OUT>, title={<TO_FILL_OUT>}, author={<TO_FILL_OUT> <TO_FILL_OUT>}, year={2025},"
        " eprint={<TO_FILL_OUT>}, archivePrefix={arXiv}, primaryClass={cs.CL} }"
    ),
    # url="",  # TODO: change to something else
    license=["<TO_FILL_OUT>"],
    version=1,
    # Bug: mlcroissant converts this to a datetime object (even when a string is specified) and then json.dumps fails
    # date_published=datetime(2025, 5, 15),
    distribution=distribution,
    record_sets=record_sets,
)

In [4]:
print(metadata.issues.report())

Found the following 1 warning(s) during the validation:
  -  [Metadata(llm_speedrun)] Property "https://schema.org/datePublished" is recommended, but does not exist.


In [5]:
import json

with open("croissant.json", "w") as f:
    content = metadata.to_json()
    content = json.dumps(content, indent=2)
    print(content)
    f.write(content)
    f.write("\n")  # Terminate file with newline

{
  "@context": {
    "@language": "en",
    "@vocab": "https://schema.org/",
    "citeAs": "cr:citeAs",
    "column": "cr:column",
    "conformsTo": "dct:conformsTo",
    "cr": "http://mlcommons.org/croissant/",
    "rai": "http://mlcommons.org/croissant/RAI/",
    "data": {
      "@id": "cr:data",
      "@type": "@json"
    },
    "dataType": {
      "@id": "cr:dataType",
      "@type": "@vocab"
    },
    "dct": "http://purl.org/dc/terms/",
    "examples": {
      "@id": "cr:examples",
      "@type": "@json"
    },
    "extract": "cr:extract",
    "field": "cr:field",
    "fileProperty": "cr:fileProperty",
    "fileObject": "cr:fileObject",
    "fileSet": "cr:fileSet",
    "format": "cr:format",
    "includes": "cr:includes",
    "isLiveDataset": "cr:isLiveDataset",
    "jsonPath": "cr:jsonPath",
    "key": "cr:key",
    "md5": "cr:md5",
    "parentField": "cr:parentField",
    "path": "cr:path",
    "recordSet": "cr:recordSet",
    "references": "cr:references",
    "regex": "cr:re

## Load rows from Croissant metadata file

In [6]:
dataset = mlc.Dataset(jsonld="croissant.json")

  -  [Metadata(llm_speedrun)] Property "https://schema.org/datePublished" is recommended, but does not exist.


In [7]:
records = dataset.records(record_set="llm_speedrun_records")

for i, record in enumerate(records):
    print(record)

{'llm_speedrun_records/id': 1, 'llm_speedrun_records/current_results': b'{\t\n\t"job_status": "COMPLETED",\n\t"metrics": {\n\t\t"n_steps": 24576,\n\t\t"val_loss": 3.2766,\n\t\t"train_time": 2968348\n\t},\n\t"hypothesis": "Baseline run of GPT2 124M model on FineWeb 10B dataset with default hyperparameters.",\n\t"outcome_summary": "The model achieves a validation loss of 3.2766 in 48.94 minutes, reaching under the 3.28 target validation loss."\n}', 'llm_speedrun_records/current_train_script': b'import os\nimport sys\nimport uuid\nimport math\nimport glob\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nimport torch._inductor.config as config\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.distributed import init_process_group, destroy_process_group\n\nwith open(sys.argv[0]) as f:\n    code = f.read()\n\n# -----------------------------------------------------------------------------\n# 